# NutriRAG: TF-IDF document retrieval

This notebook loads the ten local nutrition notes, splits them by meaningful headings and bounded text chunks, builds a TF-IDF index, and displays the top three matches for three example questions. It uses the same retrieval implementation as the app.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.knowledge_base import load_chunks
from src.retrieval import TfidfRetriever

chunks = load_chunks(PROJECT_ROOT / "data")
retriever = TfidfRetriever(chunks)
print(f"Loaded {len(list((PROJECT_ROOT / 'data').glob('*.txt')))} documents into {len(chunks)} section-aware chunks.")

Loaded 10 documents into 30 section-aware chunks.


Each chunk retains its document title and section. Text is split at section boundaries and then into overlapping word windows, so a long section can be retrieved in useful pieces.

In [2]:
import pandas as pd

def search(question, top_k=3):
    results = retriever.retrieve(question, top_k=top_k)
    return pd.DataFrame([
        {
            "document": item.chunk.document,
            "section": item.chunk.section,
            "text": item.chunk.text,
            "score": round(item.score, 4),
        }
        for item in results
    ])

search("What are good vegetarian protein sources?")

,document,section,text,score
0,Vegetarian Protein,Protein food examples,"Vegetarian protein foods include lentils, bean...",0.2617
1,Vegetarian Protein,Add protein to meals,"Include a protein-rich food in meals, such as ...",0.2489
2,Vegetarian Protein,Practical choices,"Dried or canned legumes, tofu, eggs, and dairy...",0.1666


## Test three different nutrition questions

Similarity is cosine similarity over TF-IDF vectors. A low-scoring unrelated question can return no rows because the retriever uses a minimum relevance threshold.

In [3]:
test_questions = [
    "What is a balanced diet?",
    "Suggest healthy breakfast options.",
    "How can I increase dietary fibre?",
]

for question in test_questions:
    print(f"\nQUESTION: {question}")
    display(search(question, top_k=3))


QUESTION: What is a balanced diet?


,document,section,text,score
0,Balanced Plate,Keep it realistic,"Frozen, canned, and seasonal produce can be pr...",0.1785
1,Balanced Plate,What makes a balanced plate,A balanced plate can include a variety of vege...,0.1771
2,Balanced Plate,Practical portions,"Use the plate as a flexible visual guide, not ...",0.1362



QUESTION: Suggest healthy breakfast options.


,document,section,text,score
0,Breakfast,Simple breakfast ideas,Breakfast can combine a filling grain or stapl...,0.2076
1,Breakfast,Budget-friendly choices,"Oats, eggs, seasonal fruit, and locally availa...",0.1524
2,Breakfast,Prepare ahead,"Overnight oats, boiled eggs, or pre-cut fruit ...",0.1355



QUESTION: How can I increase dietary fibre?


,document,section,text,score
0,Dietary Fibre,Fibre-containing foods,"Beans, lentils, peas, vegetables, fruits, oats...",0.4164
1,Dietary Fibre,Increasing fibre gradually,"If you want to eat more fibre, add fibre-conta...",0.2736
2,Dietary Fibre,Everyday swaps,"Try beans or lentils in a dish, choose oats or...",0.2344


Try your own query by calling `search("your nutrition question")`. Retrieved excerpts are information for general education and are not medical advice.